In [3]:
from azure.ai.ml import MLClient, automl, Input
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "e23897b8-3ae4-41ac-b1c0-d291814c906b"
RESOURCE_GROUP = "rg-mlab1-rukha21"
WORKSPACE = "mlw-lab1-rukha21"

ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE
)

print("Connected to:", ml_client.workspace_name)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Connected to: mlw-lab1-rukha21


In [4]:
job = automl.classification(
    compute="cc-lab1",
    experiment_name="knn-predictive-maintenance-automl",
    training_data=Input(
        type="mltable",
        path="azureml:ai4i-maintenance-table:1"
    ),
    target_column_name="machine_failure",
    primary_metric="AUC_weighted",
    n_cross_validations=5,
    enable_model_explainability=False,
)

job.set_limits(
    timeout_minutes=30,
    trial_timeout_minutes=15,
    max_trials=10,
    enable_early_termination=True
)

job.set_training(
    allowed_training_algorithms=["KNN"],
    enable_vote_ensemble=False,
    enable_stack_ensemble=False
)

print("AutoML KNN job configured successfully.")

AutoML KNN job configured successfully.


In [5]:
submitted = ml_client.jobs.create_or_update(job)

print("Job name:", submitted.name)
print("Status:", submitted.status)
print("Open in Studio:", submitted.studio_url)

Job name: zen_window_ys8nr8hf3g
Status: JobStatus.NOT_STARTED
Open in Studio: https://ml.azure.com/runs/zen_window_ys8nr8hf3g?wsid=/subscriptions/e23897b8-3ae4-41ac-b1c0-d291814c906b/resourcegroups/rg-mlab1-rukha21/workspaces/mlw-lab1-rukha21&tid=f7bf33fa-5901-4f76-ac74-909c195f2ea3


In [6]:
job.set_featurization(mode="off")

print("Automatic AutoML featurization is OFF.")

Automatic AutoML featurization is OFF.


In [7]:
import pandas as pd

df_automl = pd.read_csv("../data/ai4i_clean.csv")

# Encode machine type numerically for KNN compatibility
type_mapping = {
    "L": 0,
    "M": 1,
    "H": 2
}

df_automl["type"] = df_automl["type"].map(type_mapping)

print(df_automl.head())
print("\nData types:")
print(df_automl.dtypes)

print("\nMissing values:")
print(df_automl.isnull().sum())

   type  air_temp_k  process_temp_k   rpm  torque_nm  tool_wear_min  \
0     1       298.1           308.6  1551       42.8              0   
1     0       298.2           308.7  1408       46.3              3   
2     0       298.1           308.5  1498       49.4              5   
3     0       298.2           308.6  1433       39.5              7   
4     0       298.2           308.7  1408       40.0              9   

   machine_failure  
0                0  
1                0  
2                0  
3                0  
4                0  

Data types:
type                 int64
air_temp_k         float64
process_temp_k     float64
rpm                  int64
torque_nm          float64
tool_wear_min        int64
machine_failure      int64
dtype: object

Missing values:
type               0
air_temp_k         0
process_temp_k     0
rpm                0
torque_nm          0
tool_wear_min      0
machine_failure    0
dtype: int64


In [8]:
df_automl.to_csv("../data/ai4i_automl_numeric.csv", index=False)

print("Saved: ../data/ai4i_automl_numeric.csv")
print("Shape:", df_automl.shape)

Saved: ../data/ai4i_automl_numeric.csv
Shape: (10000, 7)


In [9]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

automl_data = Data(
    path="../data/ai4i_automl_numeric.csv",
    type=AssetTypes.URI_FILE,
    name="ai4i-automl-numeric",
    description="Numeric AI4I dataset for KNN AutoML compatibility"
)

registered_data = ml_client.data.create_or_update(automl_data)

print("Registered data:", registered_data.name)
print("Version:", registered_data.version)

Uploading ai4i_automl_numeric.csv (< 1 MB): 100%|██████████| 295k/295k [00:00<00:00, 7.69MB/s]




Registered data: ai4i-automl-numeric
Version: 1


In [10]:
import os
import shutil

mltable_dir = "../data/automl_mltable"
os.makedirs(mltable_dir, exist_ok=True)

# Copy the numeric CSV into the MLTable folder
shutil.copy(
    "../data/ai4i_automl_numeric.csv",
    f"{mltable_dir}/ai4i_automl_numeric.csv"
)

# Create the MLTable definition
mltable_content = """paths:
  - file: ./ai4i_automl_numeric.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: utf8
      header: all_files_same_headers
"""

with open(f"{mltable_dir}/MLTable", "w") as f:
    f.write(mltable_content)

print("MLTable folder created successfully.")
print(os.listdir(mltable_dir))

MLTable folder created successfully.
['ai4i_automl_numeric.csv', 'MLTable']


In [11]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

automl_table = Data(
    path="../data/automl_mltable",
    type=AssetTypes.MLTABLE,
    name="ai4i-automl-numeric-table",
    description="Numeric AI4I MLTable for KNN AutoML compatibility"
)

registered_table = ml_client.data.create_or_update(automl_table)

print("Registered MLTable:", registered_table.name)
print("Version:", registered_table.version)

Uploading automl_mltable (0.3 MBs): 100%|██████████| 295103/295103 [00:00<00:00, 1599162394.46it/s]




Registered MLTable: ai4i-automl-numeric-table
Version: 1


In [12]:
from azure.ai.ml import automl, Input

job_numeric = automl.classification(
    compute="cc-lab1",
    experiment_name="knn-predictive-maintenance-automl",
    training_data=Input(
        type="mltable",
        path="azureml:ai4i-automl-numeric-table:1"
    ),
    target_column_name="machine_failure",
    primary_metric="AUC_weighted",
    n_cross_validations=5,
    enable_model_explainability=False,
)

job_numeric.set_limits(
    timeout_minutes=30,
    trial_timeout_minutes=15,
    max_trials=10,
    enable_early_termination=True
)

job_numeric.set_training(
    allowed_training_algorithms=["KNN"],
    enable_vote_ensemble=False,
    enable_stack_ensemble=False
)

# Keep automatic featurization OFF because the data is already numeric
job_numeric.set_featurization(mode="off")

print("New numeric KNN AutoML job configured successfully.")

New numeric KNN AutoML job configured successfully.


In [13]:
submitted_numeric = ml_client.jobs.create_or_update(job_numeric)

print("Job name:", submitted_numeric.name)
print("Status:", submitted_numeric.status)
print("Open in Studio:", submitted_numeric.studio_url)

Job name: frosty_sugar_l7j75knhlv
Status: JobStatus.NOT_STARTED
Open in Studio: https://ml.azure.com/runs/frosty_sugar_l7j75knhlv?wsid=/subscriptions/e23897b8-3ae4-41ac-b1c0-d291814c906b/resourcegroups/rg-mlab1-rukha21/workspaces/mlw-lab1-rukha21&tid=f7bf33fa-5901-4f76-ac74-909c195f2ea3
